In [2]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:85% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:16pt;}
div.output {font-size:12pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:12pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:12px;}
</style>
"""))

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import time

%config InlineBackend.figure_format = "retina"
sns.set(style='white', context='notebook', palette='Dark2',
        rc={'figure.figsize': (12,3), 'font.family':'Malgun Gothic', 'axes.unicode_minus':False})

import warnings
warnings.filterwarnings(action='ignore')
pd.set_option('display.max_columns', None)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, recall_score, precision_score, f1_score, roc_curve


In [4]:
df = pd.read_csv('00_data/dataset_features.csv', low_memory=False)
print(df.shape)      

(254750, 164)


In [10]:
X_label = pd.read_csv('00_data/feature_cols.csv')['파생변수'].tolist()
y_label = '폐업위기_y1'

# 3절. 데이터 분할 (시간순)
- train_data : 2023Q1–2024Q4 / valid_data : 2025Q1–Q2 / test_data : 2025Q3 

In [11]:
train_data = df[df['기준_년분기_코드'] <= 20244]
valid_data = df[(df['기준_년분기_코드'] >= 20251) & (df['기준_년분기_코드'] <= 20252)]

X_train, X_val, y_train, y_val = train_data[X_label], valid_data[X_label], train_data[y_label], valid_data[y_label]

X_train.shape, X_val.shape, y_train.shape, y_val.shape

((170738, 28), (41759, 28), (170738,), (41759,))

In [12]:
print('train :', X_train.shape, ' 위기 비율', round(y_train.mean() * 100, 1), '%')
print('valid :', X_val.shape, ' 위기 비율', round(y_val.mean() * 100, 1), '%')

train : (170738, 28)  위기 비율 20.3 %
valid : (41759, 28)  위기 비율 19.3 %


## 3.1 XGBoost용 불균형 가중치  (삭제 검토필요)
- scale_pos_weight = 정상 행 수 ÷ 위기 행 수

In [13]:
spw = (y_train == 0).sum() / (y_train == 1).sum()
print('scale_pos_weight :', round(spw, 2))

scale_pos_weight : 3.92


# 4절. 모델 정의 (H0 기본값)

In [15]:
## 4.1 B1 로지스틱 회귀 
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd
X_train_oh = pd.get_dummies(X_train, columns=['상권구분', '상권변화지표', '업종'], drop_first=True, dtype=int)
X_val_oh = pd.get_dummies(X_val, columns=['상권구분', '상권변화지표', '업종'], drop_first=True, dtype=int)
X_val_oh = X_valid_oh.reindex(columns=X_train_oh.columns, fill_value=0)   # train과 컬럼 맞춤

print('원핫 전 :', X_train.shape[1], '개 → 원핫 후 :', X_train_oh.shape[1], '개')

NameError: name 'X_valid' is not defined